# Versão Final "MERIT/Hydro/v1_0_1"

In [ ]:
import os
import sys
from pathlib import Path
import gc

# --- SANITIZAÇÃO DE AMBIENTE (CRÍTICO PARA WINDOWS) ---
def sanitize_environment():
    # 1. Limpa o PATH
    original_path = os.environ.get('PATH', '')
    clean_path_list = [p for p in original_path.split(';') if 'PostgreSQL' not in p and 'PostGIS' not in p]
    os.environ['PATH'] = ';'.join(clean_path_list)
    
    # 2. Localiza PROJ interno
    venv_base = Path(sys.prefix)
    possible_paths = [
        venv_base / "Lib" / "site-packages" / "pyproj" / "proj_dir" / "share",
        venv_base / "Lib" / "site-packages" / "rasterio" / "proj_data",
        venv_base / "share" / "proj"
    ]
    
    for p in possible_paths:
        if (p / "proj.db").exists():
            os.environ['PROJ_LIB'] = str(p)
            break
            
    # Limpa variáveis órfãs
    if 'PROJ_LIB' in os.environ and not Path(os.environ['PROJ_LIB']).exists():
        del os.environ['PROJ_LIB']
    if 'GDAL_DATA' in os.environ:
        del os.environ['GDAL_DATA']

sanitize_environment()

import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pe
import seaborn as sns 
import rasterio
from rasterio.mask import mask
from rasterio.enums import Resampling
from dataclasses import dataclass, field
from matplotlib.colors import ListedColormap, LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import FuncFormatter
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.path import Path as MplPath
from shapely.geometry import Polygon, MultiPolygon
from datetime import datetime
import warnings
import unicodedata
import re

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
warnings.filterwarnings("ignore")

@dataclass
class MeritConfig:
    PATH_MUNICIPIOS: Path = Path(r"C:\Users\pedro\Downloads\python_gis\script_solos\BR_Municipios_2024\BR_Municipios_2024.shp")
    PATH_ESTADOS: Path = Path(r"C:\Users\pedro\Downloads\estagio_ie\Script_UsoeOC\BR_UF_2024\BR_UF_2024.shp")
    PATH_OUTPUT_ROOT: Path = Path(r"C:\Users\pedro\Downloads\python_gis\estudo_hidro")
    
    NOME_MUNICIPIO: str = 'Manaus'
    SIGLA_UF: str = 'AM'
    
    ASSET_ID: str = "MERIT/Hydro/v1_0_1"
    
    # Parâmetros
    RIVER_THRESHOLD_KM2: int = 5 
    SCALE_METERS: int = 90
    
    CRS_VISUAL: str = 'EPSG:3857'
    CRS_LATLON: str = 'EPSG:4674'

class GeoUtils:
    @staticmethod
    def normalize_string(text: str) -> str:
        if not isinstance(text, str): return ""
        return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8').lower().strip()

    @staticmethod
    def sanitize_filename(nome: str) -> str:
        return re.sub(r'[^\w\s-]', '', GeoUtils.normalize_string(nome)).strip().replace(' ', '_')

    @staticmethod
    def geometry_to_path(geometry):
        if isinstance(geometry, Polygon):
            return MplPath(np.array(geometry.exterior.coords))
        elif isinstance(geometry, MultiPolygon):
            paths = []
            for poly in geometry.geoms:
                paths.append(MplPath(np.array(poly.exterior.coords)))
            return MplPath.make_compound_path(*paths)
        return None

class MeritService:
    def __init__(self, config: MeritConfig):
        self.cfg = config
        self._authenticate()

    def _authenticate(self):
        try:
            ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')
        except:
            ee.Authenticate()
            ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

    def process_data(self, region_ee):
        merit = ee.Image(self.cfg.ASSET_ID)
        
        # Bandas Originais
        elev = merit.select('elv')
        upa = merit.select('upa')
        hand = merit.select('hnd')
        
        # --- CÁLCULOS AVANÇADOS ---
        # 1. Declividade (Slope) em Graus
        slope = ee.Terrain.slope(elev)
        
        # 2. Máscara de Rios
        river_mask = upa.gt(self.cfg.RIVER_THRESHOLD_KM2)
        width = merit.select('wth').updateMask(river_mask)
        
        # 3. Estatísticas Básicas (apenas para check rápido)
        stats = elev.reduceRegion(
            reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), '', True),
            geometry=region_ee.geometry(),
            scale=self.cfg.SCALE_METERS,
            maxPixels=1e13
        )
        
        # Empilhamento para exportação (5 Bandas)
        # unmask(-9999) garante que o NoData seja um valor conhecido e filtrável depois
        export_img = elev.unmask(-9999).rename('elevation') \
            .addBands(river_mask.unmask(0).rename('river_network')) \
            .addBands(hand.unmask(-9999).rename('hand')) \
            .addBands(width.unmask(0).rename('width')) \
            .addBands(slope.unmask(-9999).rename('slope'))
            
        return export_img.clip(region_ee.geometry()), stats.getInfo()

    def download_raster(self, image: ee.Image, region_ee, output_path: Path):
        if output_path.exists():
            try: os.remove(output_path)
            except: pass
            
        geemap.download_ee_image(
            image,
            filename=str(output_path),
            region=region_ee.geometry(),
            scale=self.cfg.SCALE_METERS,
            crs=self.cfg.CRS_VISUAL,
            overwrite=True,
            num_threads=4 
        )

class MeritRenderer:
    def __init__(self, config: MeritConfig):
        self.cfg = config

    def _styled_box(self, ax, title):
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor('#333333')
            spine.set_linewidth(1)
        ax.set_title(title, fontsize=9, weight='bold', pad=6, backgroundcolor='#e0e0e0', color='black')

    def _setup_grid_lines(self, ax, raster_bounds):
        minx, miny, maxx, maxy = raster_bounds
        x_target = np.linspace(minx, maxx, 4)
        y_target = np.linspace(miny, maxy, 4)
        
        pts_metric = gpd.points_from_xy(x_target, y_target, crs=self.cfg.CRS_VISUAL)
        pts_latlon = pts_metric.to_crs(self.cfg.CRS_LATLON)
        
        ax.set_xticks(x_target)
        ax.set_yticks(y_target)
        
        x_lbl = [f"{p.x:.2f}°" for p in pts_latlon]
        y_lbl = [f"{p.y:.2f}°" for p in pts_latlon]
        
        ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: x_lbl[p] if p < len(x_lbl) else ""))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda y, p: y_lbl[p] if p < len(y_lbl) else ""))
        
        ax.tick_params(axis='both', which='major', labelsize=7, direction='in', pad=4)
        ax.grid(True, linestyle='--', alpha=0.3, color='#444444', linewidth=0.5, zorder=3)

    def _draw_minimap(self, ax, mun_gdf):
        self._styled_box(ax, f"LOCALIZAÇÃO - {self.cfg.SIGLA_UF}")
        try:
            if self.cfg.PATH_ESTADOS.exists():
                states = gpd.read_file(self.cfg.PATH_ESTADOS)
                state = states[states['SIGLA_UF'] == self.cfg.SIGLA_UF.upper()]
                if not state.empty:
                    state.to_crs(self.cfg.CRS_VISUAL).plot(ax=ax, facecolor='#f4f4f4', edgecolor='#555555', linewidth=0.5)
                    mun_gdf.to_crs(self.cfg.CRS_VISUAL).plot(ax=ax, facecolor='#d62728', edgecolor='none')
            else:
                ax.text(0.5, 0.5, "Shape UF N/A", ha='center', fontsize=7)
        except: pass

    def _draw_metadata(self, ax):
        # 1. Desenha a caixa com título vazio (só borda)
        self._styled_box(ax, "")
        
        # 2. Insere "METADADOS" dentro da caixa, no topo
        ax.text(0.5, 0.95, "METADADOS", 
                ha='center', va='top', 
                fontsize=9, weight='bold', 
                transform=ax.transAxes) # backgroundcolor='#e0e0e0' opcional aqui se quiser fundo cinza no texto

        # 3. Conteúdo ajustado para ficar abaixo do título interno
        txt = (
            f"Município: {self.cfg.NOME_MUNICIPIO}\n"
            f"Dataset: JRC Global Surface Water v1.4\n"
            f"Período: 1984 - 2021\n"
            f"Resolução: {self.cfg.SCALE_METERS}m\n"
            f"Processamento: GEE & Python"
        )
        
        # 4. Renderiza o texto principal (y=0.75 para dar espaço ao título)
        ax.text(0.5, 0.75, txt, 
                ha='center', va='top', 
                fontsize=8, linespacing=1.6, 
                transform=ax.transAxes)

    def _sanitize_data(self, data_array):
        """
        CORREÇÃO CRÍTICA: Remove -inf, inf e valores de NoData (-9999)
        antes de calcular estatísticas para evitar médias quebradas.
        """
        # Cria uma cópia para não alterar o original
        clean = data_array.copy()
        
        # 1. Converte Infinitos para NaN
        clean[np.isinf(clean)] = np.nan
        
        # 2. Converte NoData conhecido para NaN (Earth Engine costuma usar -9999)
        clean[clean <= -9000] = np.nan
        
        # 3. Retorna apenas os valores finitos e válidos
        return clean[~np.isnan(clean)]

    def _draw_quick_stats(self, ax, data_array, unit):
        self._styled_box(ax, "RESUMO")
        
        # Usa a função de limpeza blindada
        valid = self._sanitize_data(data_array)
        
        if valid.size == 0:
            ax.text(0.5, 0.5, "Sem dados válidos", ha='center', fontsize=8)
            return
            
        mn, mx, me = np.min(valid), np.max(valid), np.mean(valid)
        std = np.std(valid)
        
        txt = (
            f"Mínimo: {mn:.1f} {unit}\n"
            f"Máximo: {mx:.1f} {unit}\n"
            f"Média:  {me:.1f} {unit}\n"
            f"Desvio: {std:.1f}"
        )
        ax.text(0.5, 0.5, txt, fontsize=8, ha='center', va='center', fontfamily='monospace', linespacing=1.6)

    def generate_map(self, raster_path, aoi_gdf, map_type='elevation'):
        fig = plt.figure(figsize=(11.69, 8.27))
        gs = gridspec.GridSpec(1, 2, width_ratios=[3, 1], wspace=0.1, figure=fig)
        
        ax_map = fig.add_subplot(gs[0])
        
        # Configuração do Mapa
        if map_type == 'elevation':
            band_idx = 1
            cmap = 'terrain'
            title = "ALTIMETRIA DIGITAL"
            unit = "(m)"
            vmin, vmax = None, None
        elif map_type == 'slope':
            band_idx = 5 # Nova banda de Slope
            cmap = 'magma_r'
            title = "DECLIVIDADE (RELEVO)"
            unit = "Graus (°)"
            vmin, vmax = 0, 30 
        elif map_type == 'hand':
            band_idx = 3
            cmap = 'Blues_r'
            title = "MODELO HAND (RISCO HÍDRICO)"
            unit = "(m)"
            vmin, vmax = 0, 20
        elif map_type == 'network':
            band_idx = 4
            cmap = 'GnBu'
            title = "REDE DE DRENAGEM ESTIMADA"
            unit = "(m)"
            vmin, vmax = None, None

        with rasterio.open(raster_path) as src:
            overscale = max(1, src.width // 2000)
            data = src.read(
                band_idx,
                out_shape=(src.height // overscale, src.width // overscale),
                resampling=Resampling.nearest
            )
            
            # Limpeza visual (Mascara -9999 para transparente)
            data = data.astype(float) # Garante float para aceitar NaN
            data[data <= -9000] = np.nan # Filtra NoData visualmente
            
            if map_type == 'network':
                data[data <= 0] = np.nan # Rios com largura 0 somem
                
            extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)
            
            im = ax_map.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, extent=extent, zorder=1)
            self._setup_grid_lines(ax_map, src.bounds)

        # Vetor
        aoi_proj = aoi_gdf.to_crs(self.cfg.CRS_VISUAL)
        aoi_proj.plot(ax=ax_map, facecolor='none', edgecolor='black', linewidth=1.5, zorder=2)
        
        ax_map.add_artist(ScaleBar(1, units='m', location='lower left', box_alpha=0.7, font_properties={'family': 'serif', 'size': 8}))
        
        ax_map.annotate('N', xy=(0.95, 0.95), xytext=(0.95, 0.88), 
                        arrowprops=dict(facecolor='black', width=4, headwidth=10),
                        ha='center', va='center', fontsize=12, weight='bold', xycoords='axes fraction', zorder=10)
        
        ax_map.set_title(f"{title}\n{self.cfg.NOME_MUNICIPIO} - {self.cfg.SIGLA_UF}", fontsize=14, weight='bold', pad=15)

        # Lateral
        gs_side = gridspec.GridSpecFromSubplotSpec(4, 1, subplot_spec=gs[1], height_ratios=[0.25, 0.1, 0.25, 0.4], hspace=0.3)
        self._draw_minimap(fig.add_subplot(gs_side[0]), aoi_gdf)
        
        ax_leg = fig.add_subplot(gs_side[1])
        self._styled_box(ax_leg, "LEGENDA")
        cbar = plt.colorbar(im, cax=ax_leg, orientation='horizontal', fraction=0.5, pad=0.1)
        cbar.set_label(unit, fontsize=8, weight='bold')

        self._draw_metadata(fig.add_subplot(gs_side[2]))
        
        # Stats corrigido com _sanitize_data
        self._draw_quick_stats(fig.add_subplot(gs_side[3]), data, unit.split(' ')[0])
        
        return fig

    def generate_dashboard(self, raster_path):
        """Dashboard Estatístico Avançado (Curva Hipsométrica, Slope Classes)"""
        fig = plt.figure(figsize=(11.69, 8.27))
        fig.suptitle(f"ANÁLISE MORFOMÉTRICA E HIDROLÓGICA - {self.cfg.NOME_MUNICIPIO}", fontsize=16, weight='bold', y=0.95)
        
        gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1], hspace=0.3, wspace=0.3)
        
        # Leitura dos dados
        with rasterio.open(raster_path) as src:
            scale = max(1, src.width // 1000)
            elev = src.read(1, out_shape=(src.height//scale, src.width//scale), resampling=Resampling.bilinear)
            hand = src.read(3, out_shape=(src.height//scale, src.width//scale), resampling=Resampling.nearest)
            slope = src.read(5, out_shape=(src.height//scale, src.width//scale), resampling=Resampling.bilinear)
            
            # Sanitização Robusta
            elev = self._sanitize_data(elev)
            hand = self._sanitize_data(hand)
            slope = self._sanitize_data(slope)

        if elev.size == 0 or slope.size == 0:
            ax_err = fig.add_subplot(111); ax_err.text(0.5,0.5, "Erro: Sem dados válidos no raster"); ax_err.axis('off')
            return fig

        # 1. Curva Hipsométrica
        ax_hyp = fig.add_subplot(gs[0, 0])
        sorted_elev = np.sort(elev)
        y_vals = np.linspace(0, 100, len(sorted_elev))
        ax_hyp.plot(sorted_elev, y_vals, color='brown', linewidth=2)
        ax_hyp.fill_between(sorted_elev, y_vals, color='orange', alpha=0.3)
        ax_hyp.set_title("Curva Hipsométrica (Altimetria)", weight='bold', fontsize=10)
        ax_hyp.set_xlabel("Altitude (m)"); ax_hyp.set_ylabel("% Área Acumulada")
        ax_hyp.grid(True, linestyle='--', alpha=0.5)

        # 2. Histograma de Declividade (Slope)
        ax_slope = fig.add_subplot(gs[0, 1])
        sns.histplot(slope, kde=True, ax=ax_slope, color='purple', bins=30, stat="percent")
        ax_slope.set_title("Distribuição de Declividade", weight='bold', fontsize=10)
        ax_slope.set_xlabel("Inclinação (Graus)"); ax_slope.set_ylabel("% da Área")
        ax_slope.set_xlim(0, 45) 
        
        # 3. Classificação de Risco HAND (Pie Chart)
        ax_risk = fig.add_subplot(gs[1, 0])
        high = np.sum(hand < 5)
        med = np.sum((hand >= 5) & (hand < 15))
        low = np.sum(hand >= 15)
        total = len(hand)
        
        if total > 0:
            sizes = [high, med, low]
            labels = [f'Alto Risco (<5m)\n{high/total:.1%}', 
                      f'Médio (5-15m)\n{med/total:.1%}', 
                      f'Baixo (>15m)\n{low/total:.1%}']
            colors = ['#ff6666', '#ffcc00', '#99ff99']
            ax_risk.pie(sizes, labels=labels, colors=colors, startangle=90, wedgeprops={'edgecolor': 'black'})
        ax_risk.set_title("Potencial de Inundação (Modelo HAND)", weight='bold', fontsize=10)

        # 4. Boxplot Comparativo
        ax_box = fig.add_subplot(gs[1, 1])
        data_viz = [elev, slope*10] 
        ax_box.boxplot(data_viz, labels=['Altitude (m)', 'Declividade (x10)'], patch_artist=True,
                       boxprops=dict(facecolor='lightblue'))
        ax_box.set_title("Dispersão de Dados", weight='bold', fontsize=10)
        ax_box.grid(True, axis='y', linestyle='--', alpha=0.5)
        
        return fig

class MeritPipeline:
    def __init__(self, config: MeritConfig):
        self.cfg = config
        self.service = MeritService(config)
        self.renderer = MeritRenderer(config)

    def run(self):
        temp_tif = None
        try:
            print(f"--- INICIANDO MERIT: {self.cfg.NOME_MUNICIPIO} ---")
            
            # 1. Carrega Vetor
            full_gdf = gpd.read_file(self.cfg.PATH_MUNICIPIOS)
            full_gdf['norm_nm'] = full_gdf['NM_MUN'].apply(GeoUtils.normalize_string)
            aoi = full_gdf[(full_gdf['norm_nm'] == GeoUtils.normalize_string(self.cfg.NOME_MUNICIPIO)) & 
                           (full_gdf['SIGLA_UF'] == self.cfg.SIGLA_UF.upper())].dissolve()
            
            if aoi.empty: raise ValueError("Município não encontrado.")
            
            print("1. Processando MERIT Hydro + Declividade...")
            aoi_ee = geemap.geopandas_to_ee(aoi.to_crs("EPSG:4326"))
            img_hydro, _ = self.service.process_data(aoi_ee)
            
            safe_name = GeoUtils.sanitize_filename(self.cfg.NOME_MUNICIPIO)
            temp_tif = self.cfg.PATH_OUTPUT_ROOT / f"TEMP_MERIT_{safe_name}.tif"
            self.cfg.PATH_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
            
            print("2. Baixando Raster Multibanda (Tiled)...")
            self.service.download_raster(img_hydro, aoi_ee, temp_tif)
            
            if not temp_tif.exists(): raise FileNotFoundError("Erro no download.")
            
            print("3. Gerando Relatório Avançado...")
            figs = [
                self.renderer.generate_map(temp_tif, aoi, 'elevation'),
                self.renderer.generate_map(temp_tif, aoi, 'slope'),    
                self.renderer.generate_map(temp_tif, aoi, 'hand'),
                self.renderer.generate_map(temp_tif, aoi, 'network'),
                self.renderer.generate_dashboard(temp_tif)             
            ]
            
            out_pdf = self.cfg.PATH_OUTPUT_ROOT / f"RELATORIO_TOPOGRAFIA_{safe_name}.pdf"
            with PdfPages(out_pdf) as pdf:
                for f in figs:
                    pdf.savefig(f, bbox_inches='tight', dpi=300)
                    plt.close(f)
            
            print(f" SUCESSO: {out_pdf}")
            
        except Exception as e:
            print(f" ERRO: {e}")
            import traceback; traceback.print_exc()
        finally:
            if temp_tif and temp_tif.exists():
                try: 
                    gc.collect()
                    temp_tif.unlink()
                except: pass

if __name__ == "__main__":
    cfg = MeritConfig(NOME_MUNICIPIO='Três Corações', SIGLA_UF='MG')
    MeritPipeline(cfg).run()